# 📊 Notebook 1 — Data Cleaning & Preparation

This notebook loads the two raw datasets, cleans them, handles missing values,
and merges them into one master DataFrame for use in all subsequent notebooks.

**Datasets:**
- `loadshedding.csv` — Daily Eskom load shedding stage and hours (Jan–Mar 2024)
- `revenue.csv` — Daily revenue for a Johannesburg retail shop

**Goal:** Produce `data/merged.csv` connecting each day's load shedding data to its revenue.

In [ ]:
import pandas as pd
import numpy as np
import sys, os
print('Libraries loaded ✅')

## Step 1 — Load Raw Data

We parse dates immediately so pandas can sort, filter, and group by date correctly.

In [ ]:
ls_df  = pd.read_csv('../data/loadshedding.csv', parse_dates=['date'])
rev_df = pd.read_csv('../data/revenue.csv',     parse_dates=['date'])
print('Load shedding shape:', ls_df.shape)
print('Revenue shape:      ', rev_df.shape)
ls_df.head()

In [ ]:
rev_df.head()

## Step 2 — Inspect Missing Values

Days with Stage 0 (no load shedding) have no scheduled start/end times. These nulls are expected — not errors.

In [ ]:
print('Missing — load shedding:')
print(ls_df.isnull().sum())
print()
print('Missing — revenue:')
print(rev_df.isnull().sum())

We fill the null start/end times with the string `'None'` to make them explicit.

In [ ]:
ls_df['scheduled_start'] = ls_df['scheduled_start'].fillna('None')
ls_df['scheduled_end']   = ls_df['scheduled_end'].fillna('None')
ls_df['stage']           = ls_df['stage'].fillna(0).astype(int)
ls_df['hours_affected']  = ls_df['hours_affected'].fillna(0)
print('Nulls after cleaning:', ls_df.isnull().sum().sum())

## Step 3 — Derive New Columns

- `stage_category` groups stages into Low / Medium / High for cleaner charts
- `is_loadshedding` is a boolean flag for any day with load shedding
- `month`, `week`, `day_name` enable time-period grouping

In [ ]:
def categorise_stage(s):
    if s == 0:   return 'None'
    elif s <= 2: return 'Low (1-2)'
    elif s <= 4: return 'Medium (3-4)'
    else:        return 'High (5-6)'

ls_df['stage_category']  = ls_df['stage'].apply(categorise_stage)
ls_df['is_loadshedding'] = ls_df['stage'] > 0

rev_df['month']     = rev_df['date'].dt.month_name()
rev_df['month_num'] = rev_df['date'].dt.month
rev_df['week']      = rev_df['date'].dt.to_period('W').astype(str)
rev_df['day_name']  = rev_df['date'].dt.day_name()

ls_df[['date','stage','stage_category','is_loadshedding']].head(10)

## Step 4 — Merge the Datasets

We use a **LEFT JOIN** on date — keeping every revenue row and attaching load shedding data where available. Any unmatched dates get filled with Stage 0.

In [ ]:
merged = pd.merge(
    rev_df,
    ls_df[['date','stage','hours_affected','stage_category','is_loadshedding']],
    on='date', how='left'
)
merged['stage']           = merged['stage'].fillna(0).astype(int)
merged['hours_affected']  = merged['hours_affected'].fillna(0)
merged['stage_category']  = merged['stage_category'].fillna('None')
merged['is_loadshedding'] = merged['is_loadshedding'].fillna(False)
print('Merged shape:', merged.shape)
merged[['date','revenue','stage','hours_affected','stage_category']].head(10)

## Step 5 — Calculate Baseline Revenue & Revenue Loss

The **baseline** is the average daily revenue on normal trading days — no load shedding, no public holidays, shop open.

**Revenue loss per day** = baseline − actual revenue (only on load shedding days).

In [ ]:
normal_days = merged[
    (merged['stage'] == 0) &
    (merged['is_public_holiday'] == False) &
    (merged['revenue'] > 0)
]
baseline = normal_days['revenue'].mean()
print(f'Baseline daily revenue: R {baseline:,.2f}')
print(f'Normal trading days used: {len(normal_days)}')

merged['baseline_revenue'] = baseline
merged['revenue_loss'] = merged.apply(
    lambda r: max(0, baseline - r['revenue'])
    if r['is_loadshedding'] and not r['is_public_holiday'] else 0, axis=1
)
merged['pct_of_baseline'] = merged.apply(
    lambda r: (r['revenue'] / baseline) * 100
    if r['revenue'] > 0 and not r['is_public_holiday'] else None, axis=1
)
print(f'Total revenue lost to load shedding: R {merged["revenue_loss"].sum():,.2f}')

## Step 6 — Export Clean Data

Save to `data/merged.csv` so all other notebooks can load it directly.

In [ ]:
merged.to_csv('../data/merged.csv', index=False)
print('✅ Exported to data/merged.csv')
merged[['revenue','stage','hours_affected','revenue_loss']].describe().round(2)